(ref_tutorial_FLE)=
# VTK Tutorial 12: Fiducial Localization Error

In this ($12^{th}$) tutorial, we are going to investigate how Fiducial Localization Error ({term}`FLE`) influence the rigid-body registration accuracy.

**More examples and tutorials can be found [here](https://examples.vtk.org/site/Python/)**

To begin with, we need to import all the necessary Python functions and external libraries:

## Fiducial Localization Errors

[We previously discussed how to error in registration is a direct consequence of Fiducial Localization Error ({term}`FLE`)](#ref_Registration_Errors). Here, we are going visualize this process.

## Python Setup

Most of the methods for manipulating this transformation, e.g. Translate, Rotate, and Concatenate, can operate in either **PreMultiply** (the default) or **PostMultiply** mode. In PreMultiply mode, the translation, concatenation, etc. will occur before any transformations which are represented by the current matrix. In **PostMultiply** mode, the additional transformation will occur after any transformations represented by the current matrix.

The [presentation we use](#ref_transforms_chap) adheres to the PostMultiply mode.

In [ ]:
import math
import numpy as np

# noinspection PyUnresolvedReferences
import vtkmodules.vtkInteractionStyle
# noinspection PyUnresolvedReferences
import vtkmodules.vtkRenderingOpenGL2
from vtkmodules.vtkCommonColor import vtkNamedColors
from vtkmodules.vtkCommonTransforms import vtkTransform
from vtkmodules.vtkFiltersSources import vtkSphereSource
from vtkmodules.vtkRenderingAnnotation import vtkAxesActor
from vtkmodules.vtkCommonCore import vtkPoints
from vtkmodules.vtkCommonTransforms import vtkLandmarkTransform
from vtkmodules.vtkRenderingCore import (
    vtkActor,
    vtkPolyDataMapper,
    vtkRenderWindow,
    vtkRenderWindowInteractor,
    vtkRenderer
)

Notice how ```vtkPoints``` is imported.

```python
from vtkmodules.vtkCommonCore import vtkPoints
```

```vtkLandmarkTransform``` is ```VTK```'s implementation of the {term}`OPA` solution.

```python
from vtkmodules.vtkCommonTransforms import vtkLandmarkTransform
```

## Rendering

In [29]:
# Visualization Pipeline

# a renderer and render window
ren = vtkRenderer()
renWin = vtkRenderWindow()
renWin.SetWindowName('AISE 4025: Transformations')
renWin.SetSize( 640, 480 )
renWin.AddRenderer( ren )

# an interactor
iren = vtkRenderWindowInteractor()
iren.SetRenderWindow( renWin )

## Geometry

Using this [previous example of optical tracking as the context](#ref_Tracking), let's define 4 *registration fiducial points* (i.e. the location of the retroreflective spheres) and a single *target fiducial point* (i.e. tip of the stylus):

| Marker | x (mm) | y (mm) | z (mm) |
|:--------:|---------:|---------:|---------:|
| A   | 0.00   |  0.00   |    0.00 |
| B   | 0.00   |  0.00   |   50.00 |
| C   | 0.00   | 25.00   |  100.00 |
| D   | 0.00   | -25.00  |  135.00 |
| tip   | 10.00   | 0.00  |  -150.00 |

In [30]:
colors = vtkNamedColors()

RegistrationFiducial = vtkPoints()
RegistrationFiducial.SetDataTypeToDouble()

RegistrationFiducial.InsertNextPoint( 0, 0, 0 )
RegistrationFiducial.InsertNextPoint( 0, 0, 50 )
RegistrationFiducial.InsertNextPoint( 0, 25, 100 )
RegistrationFiducial.InsertNextPoint( 0, -25, 135 )

TargetFiducial = vtkPoints()
TargetFiducial.InsertNextPoint( 10, 0, -150 )

0

Define a transformation. For the purpose of this tutorial, the exact transformation does not matter as long as it is not identity:

In [31]:
alpha = 60 # 60 degree
beta = 45
transform = vtkTransform()
transform.PostMultiply()
transform.Identity()
transform.RotateX( alpha )
transform.RotateY( beta )
transform.Translate( 1, 2, 3)
transform.Update()
print( "\nThe ground truth transformation is:\n", transform.GetMatrix() )


The ground truth transformation is:
 vtkMatrix4x4 (0000017E5C2F1AE0)
  Debug: Off
  Modified Time: 3428
  Reference Count: 2
  Registered Events: (none)
  Elements:
    0.707107 0.612372 0.353553 1 
    0 0.5 -0.866025 2 
    -0.707107 0.612372 0.353553 3 
    0 0 0 1 




Perform the transformation to both the registration and target fiducial points:

In [32]:
transformedRegistrationFiducial = vtkPoints()
transformedTargetFiducial = vtkPoints()

transform.TransformPoints( RegistrationFiducial, transformedRegistrationFiducial )
transform.TransformPoints( TargetFiducial, transformedTargetFiducial )

## Ground Truth

Now we have established a ground truth, i.e. we explicitly specified a **known** transformation, and applied the model points with the transformation without any errors.

Let's print these ground truth on the screen.

In [33]:
print( "The model registration fiducial points are: \n")
for i in range(RegistrationFiducial.GetNumberOfPoints() ):
    print( RegistrationFiducial.GetPoint(i) )
print( "\nThe error-free transformed fiducial points are: \n" )
for i in range( transformedRegistrationFiducial.GetNumberOfPoints() ):
    print(transformedRegistrationFiducial.GetPoint(i))

print( "\nThe model target fiducial point is: \n" )
print( TargetFiducial.GetPoint(0) )
print( "\nThe transformed target fiducial point is: \n" )
print(transformedTargetFiducial.GetPoint(0))

print( "These are the ground truth!")

The model registration fiducial points are: 

(0.0, 0.0, 0.0)
(0.0, 0.0, 50.0)
(0.0, 25.0, 100.0)
(0.0, -25.0, 135.0)

The error-free transformed fiducial points are: 

(1.0, 2.0, 3.0)
(18.677669525146484, -41.30126953125, 20.677669525146484)
(51.664649963378906, -72.1025390625, 53.664649963378906)
(33.4203987121582, -127.4134292602539, 35.4203987121582)

The model target fiducial point is: 

(10.0, 0.0, -150.0)

The transformed target fiducial point is: 

(-44.96194076538086, 131.90380859375, -57.10407638549805)
These are the ground truth!


## Transform via {term}`OPA`

Given the model and measured fiducial point sets, can we recover the underlying transformation?

In [ ]:
OPA = vtkLandmarkTransform()
OPA.SetSourceLandmarks( RegistrationFiducial )
OPA.SetTargetLandmarks( transformedRegistrationFiducial )
OPA.SetModeToRigidBody()
OPA.Update()
print( OPA.GetMatrix() )

vtkMatrix4x4 (0000017E5BD7F7F0)
  Debug: Off
  Modified Time: 3458
  Reference Count: 2
  Registered Events: (none)
  Elements:
    0.707107 0.612372 0.353553 1 
    0 0.5 -0.866025 2 
    -0.707107 0.612372 0.353553 3 
    0 0 0 1 




Indeed we can recover the underlying transformation using ```vtkLandmarkTransform```!

## Introducing Fiducial Localization Errors

Let's **contaminate** the measured point fiducial with some {term}`FLE`. Again, for the purpose of this tutorial, the exact magnitude does not matter.

We will utilize ```np.random.normal``` to generate random numbers. You should understand what the parameters are!

In [54]:
measuredFiducial = vtkPoints()

for i in range( transformedRegistrationFiducial.GetNumberOfPoints() ):
    point = transformedRegistrationFiducial.GetPoint( i )
    FLE = np.random.normal( 0, 1.5, 3 )
    measuredFiducial.InsertNextPoint( point+FLE )

for i in range( measuredFiducial.GetNumberOfPoints() ):
    print( measuredFiducial.GetPoint(i) )
    print( transformedRegistrationFiducial.GetPoint(i), "\n" ) 

(-0.14315322041511536, 1.54804527759552, 5.932057857513428)
(1.0, 2.0, 3.0) 

(16.19550323486328, -40.0852165222168, 18.875940322875977)
(18.677669525146484, -41.30126953125, 20.677669525146484) 

(51.440269470214844, -69.43252563476562, 53.786277770996094)
(51.664649963378906, -72.1025390625, 53.664649963378906) 

(33.595848083496094, -127.14561462402344, 37.50246810913086)
(33.4203987121582, -127.4134292602539, 35.4203987121582) 



### Transformation with FLE?

Now our measurements are contaminated with {term}`FLE`, can we still recover the underlying transformation? If so, how do we evaluate its **fitness**?

In [55]:
OPA_FLE = vtkLandmarkTransform()
OPA_FLE.SetSourceLandmarks( RegistrationFiducial )
OPA_FLE.SetTargetLandmarks( measuredFiducial )
OPA_FLE.SetModeToRigidBody()
OPA_FLE.Update()
print( OPA_FLE.GetMatrix() )

vtkMatrix4x4 (0000017E59F57430)
  Debug: Off
  Modified Time: 3561
  Reference Count: 2
  Registered Events: (none)
  Elements:
    0.681402 0.629307 0.373718 -1.35527 
    -0.00417824 0.513943 -0.857814 2.34045 
    -0.731898 0.582955 0.352831 3.88499 
    0 0 0 1 




## Target Registration Error

We now see that the transformation we derived using the contaminated measurements is different from the ground truth, which is expected, but how good (or how bad) is this registration?

One way to quantify is via Target Registration Error ({term}`TRE`).

In [67]:
transformedTargetFiducial_with_FLE = vtkPoints()

OPA_FLE.TransformPoints( TargetFiducial, transformedTargetFiducial_with_FLE )

print( "The TRE is: \n", 
      np.linalg.norm( np.array(transformedTargetFiducial.GetPoint(0)) - 
                     np.array(transformedTargetFiducial_with_FLE.GetPoint(0))))



The TRE is: 
 5.7620700337283175


## Final Thoughts

You should perhaps wrap this calculation of target registration error in a loop and run it through, perhaps, thousands of times and calculate the **mean** {term}`TRE` instead. Based on the <wiki:Law_of_large_numbers>, what do you expect the mean {trem}`TRE` to be?

Unfortunately, during the surgery, we don't have the luxury of running multiple trials of registration!  Thus, we need to find a way to estimate TRE reliably.